# Chapter 8.1: Model Registration System in vLLM

## Learning Objectives

By the end of this notebook, you will:

1. **Understand** how vLLM discovers and loads model architectures
2. **Master** the ModelRegistry and its internal mechanics
3. **Learn** the registration decorators and entry points
4. **Know** the required model class interface (methods, attributes)
5. **Understand** ModelConfig and its mapping to HuggingFace config
6. **Implement** architecture name mapping
7. **Declare** supported features (LoRA, Pipeline Parallelism, Quantization)
8. **Walk through** the actual vLLM source code for model registration
9. **Register** a toy model from scratch

---

## Prerequisites
- Familiarity with PyTorch `nn.Module`
- Basic understanding of transformer architectures
- Python class inheritance and decorators

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part8/chapter_8.1_vllm_registration.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part8/chapter_8.1_vllm_registration.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. Overview: How vLLM Finds Your Model

When you call `LLM(model="meta-llama/Llama-3-8B")`, vLLM needs to:

1. Download the HuggingFace config (`config.json`)
2. Read the `architectures` field (e.g., `["LlamaForCausalLM"]`)
3. Look up `LlamaForCausalLM` in its **ModelRegistry**
4. Import the corresponding Python class
5. Instantiate it with the proper configuration
6. Load the weights

The **ModelRegistry** is the central mechanism that maps architecture names to Python classes.

```
                  HuggingFace config.json
                         |
                         v
              architectures: ["LlamaForCausalLM"]
                         |
                         v
              +-------------------+
              |   ModelRegistry   |
              |  (name -> class)  |
              +-------------------+
                         |
                         v
              vllm.model_executor.models.llama.LlamaForCausalLM
                         |
                         v
              Model instantiation + weight loading
```

## 2. The ModelRegistry: Core Data Structure

The `ModelRegistry` in vLLM is defined in `vllm/model_executor/models/registry.py`. It maintains a dictionary mapping architecture names (strings) to module paths and class names.

Let's examine the structure:

In [ ]:
# Simulating the vLLM ModelRegistry structure
# In actual vLLM source, this is in vllm/model_executor/models/registry.py

# The registry maps architecture names to (module_path, class_name) tuples
# Here's a simplified version of what the registry looks like:

_VLLM_MODELS = {
    # Architecture name -> (module_name, class_name)
    "LlamaForCausalLM": ("llama", "LlamaForCausalLM"),
    "MistralForCausalLM": ("llama", "LlamaForCausalLM"),  # Mistral reuses Llama!
    "GPT2LMHeadModel": ("gpt2", "GPT2LMHeadModel"),
    "Qwen2ForCausalLM": ("qwen2", "Qwen2ForCausalLM"),
    "GemmaForCausalLM": ("gemma", "GemmaForCausalLM"),
    "Gemma2ForCausalLM": ("gemma2", "Gemma2ForCausalLM"),
    "PhiForCausalLM": ("phi", "PhiForCausalLM"),
    "LlavaForConditionalGeneration": ("llava", "LlavaForConditionalGeneration"),
    # ... hundreds more entries
}

print(f"Number of registered architectures (sample): {len(_VLLM_MODELS)}")
for arch, (module, cls) in _VLLM_MODELS.items():
    print(f"  {arch} -> vllm.model_executor.models.{module}.{cls}")

## 3. Source Code Walkthrough: registry.py

Let's look at the actual structure of the registry in vLLM's source code. The key file is `vllm/model_executor/models/registry.py`.

In [ ]:
# Actual vLLM registry structure (simplified from source)
# File: vllm/model_executor/models/registry.py

from dataclasses import dataclass, field
from typing import Dict, Optional, Type, Callable, Tuple, List

@dataclass
class _ModelInfo:
    """Information about a registered model."""
    # The architecture name as it appears in HuggingFace config
    architecture_name: str
    # Module path relative to vllm.model_executor.models
    module_name: str
    # Class name within the module
    class_name: str
    # Whether this model supports LoRA
    supports_lora: bool = False
    # Whether this model supports pipeline parallelism
    supports_pp: bool = False
    # Whether this model is multimodal
    is_multimodal: bool = False
    
    def get_full_module_path(self) -> str:
        return f"vllm.model_executor.models.{self.module_name}"

# Example entries
info = _ModelInfo(
    architecture_name="LlamaForCausalLM",
    module_name="llama",
    class_name="LlamaForCausalLM",
    supports_lora=True,
    supports_pp=True,
    is_multimodal=False
)
print(f"Model: {info.architecture_name}")
print(f"Full path: {info.get_full_module_path()}.{info.class_name}")
print(f"LoRA: {info.supports_lora}, PP: {info.supports_pp}, Multimodal: {info.is_multimodal}")

In [ ]:
# The actual _VLLM_MODELS dictionary uses a more compact format.
# In vLLM's source code, it looks like this:

# Each entry is:
#   "ArchitectureName": ("module_name", "ClassName"),
#
# The module_name is relative to vllm/model_executor/models/
# So "llama" means vllm/model_executor/models/llama.py

_MODELS = {
    # === Text-only decoder models ===
    "AquilaForCausalLM": ("llama", "LlamaForCausalLM"),
    "BaiChuanForCausalLM": ("baichuan", "BaiChuanForCausalLM"),
    "BloomForCausalLM": ("bloom", "BloomForCausalLM"),
    "ChatGLMModel": ("chatglm", "ChatGLMForCausalLM"),
    "DeepseekForCausalLM": ("deepseek", "DeepseekForCausalLM"),
    "DeepseekV2ForCausalLM": ("deepseek_v2", "DeepseekV2ForCausalLM"),
    "FalconForCausalLM": ("falcon", "FalconForCausalLM"),
    "GemmaForCausalLM": ("gemma", "GemmaForCausalLM"),
    "GPTBigCodeForCausalLM": ("gpt_bigcode", "GPTBigCodeForCausalLM"),
    "GPT2LMHeadModel": ("gpt2", "GPT2LMHeadModel"),
    "GPTJForCausalLM": ("gpt_j", "GPTJForCausalLM"),
    "GPTNeoXForCausalLM": ("gpt_neox", "GPTNeoXForCausalLM"),
    "LlamaForCausalLM": ("llama", "LlamaForCausalLM"),
    "MistralForCausalLM": ("llama", "LlamaForCausalLM"),  # Reuses Llama!
    "MixtralForCausalLM": ("mixtral", "MixtralForCausalLM"),
    "PhiForCausalLM": ("phi", "PhiForCausalLM"),
    "Phi3ForCausalLM": ("phi3", "Phi3ForCausalLM"),
    "Qwen2ForCausalLM": ("qwen2", "Qwen2ForCausalLM"),
    "Qwen2MoeForCausalLM": ("qwen2_moe", "Qwen2MoeForCausalLM"),
    "StableLMForCausalLM": ("stablelm", "StableLmForCausalLM"),
    
    # === Multimodal models ===
    "LlavaForConditionalGeneration": ("llava", "LlavaForConditionalGeneration"),
    "LlavaNextForConditionalGeneration": ("llava_next", "LlavaNextForConditionalGeneration"),
}

print(f"Total registered: {len(_MODELS)} architectures")
print("\nNote: Multiple architectures can map to the same class!")
print(f"  MistralForCausalLM -> {_MODELS['MistralForCausalLM']}")
print(f"  LlamaForCausalLM -> {_MODELS['LlamaForCausalLM']}")
print("  Both use the same implementation!")

## 4. The ModelRegistry Class

The `ModelRegistry` class provides the API for looking up and registering models. Let's examine its key methods.

In [ ]:
import importlib
from typing import Dict, Optional, Type, Tuple

class ModelRegistry:
    """
    Simplified version of vLLM's ModelRegistry.
    
    The actual implementation is in vllm/model_executor/models/registry.py
    Key responsibilities:
    1. Store architecture -> class mappings
    2. Lazy import of model classes (for fast startup)
    3. Support for user-registered custom models
    4. Validation of model capabilities
    """
    
    def __init__(self):
        # Built-in models
        self._models: Dict[str, Tuple[str, str]] = {}
        # User-registered models (class objects, not lazy imports)
        self._user_models: Dict[str, Type] = {}
    
    def register_model(self, arch_name: str, module_name: str, class_name: str):
        """Register a built-in model with lazy loading."""
        self._models[arch_name] = (module_name, class_name)
        print(f"Registered: {arch_name} -> {module_name}.{class_name}")
    
    def register_user_model(self, arch_name: str, model_class: Type):
        """Register a user-provided model class directly."""
        self._user_models[arch_name] = model_class
        print(f"User-registered: {arch_name} -> {model_class.__name__}")
    
    def resolve_model_cls(self, architecture: str) -> Type:
        """
        Resolve an architecture name to a model class.
        
        Priority:
        1. User-registered models (override built-in)
        2. Built-in models (lazy imported)
        """
        # Check user models first
        if architecture in self._user_models:
            return self._user_models[architecture]
        
        # Check built-in models
        if architecture in self._models:
            module_name, class_name = self._models[architecture]
            full_module = f"vllm.model_executor.models.{module_name}"
            module = importlib.import_module(full_module)
            return getattr(module, class_name)
        
        raise ValueError(
            f"Model architecture '{architecture}' is not supported. "
            f"Supported: {list(self._models.keys()) + list(self._user_models.keys())}"
        )
    
    def is_supported(self, architecture: str) -> bool:
        """Check if an architecture is supported."""
        return architecture in self._models or architecture in self._user_models
    
    def list_models(self) -> list:
        """List all registered architecture names."""
        return sorted(set(list(self._models.keys()) + list(self._user_models.keys())))

# Demo
registry = ModelRegistry()
registry.register_model("LlamaForCausalLM", "llama", "LlamaForCausalLM")
registry.register_model("GPT2LMHeadModel", "gpt2", "GPT2LMHeadModel")
print(f"\nIs Llama supported? {registry.is_supported('LlamaForCausalLM')}")
print(f"Is BERT supported? {registry.is_supported('BertModel')}")
print(f"All models: {registry.list_models()}")

## 5. Lazy Loading: Why It Matters

vLLM supports 100+ model architectures. If it imported all model classes eagerly at startup, it would:
- Import dozens of heavy modules
- Increase startup time significantly
- Consume unnecessary memory

Instead, vLLM uses **lazy loading**: it only imports the model class when it's actually needed.

In [ ]:
import time

# Demonstrate the difference between eager and lazy loading

class LazyModelRegistry:
    """Registry with lazy loading - only imports when needed."""
    
    def __init__(self):
        # Store as strings, not actual classes
        self._registry: Dict[str, Tuple[str, str]] = {}
        self._cache: Dict[str, Type] = {}  # Cache resolved classes
    
    def register(self, arch: str, module: str, cls: str):
        self._registry[arch] = (module, cls)
    
    def get_model_class(self, arch: str) -> Type:
        """Lazy import: only loads the module when actually needed."""
        if arch in self._cache:
            return self._cache[arch]
        
        if arch not in self._registry:
            raise KeyError(f"Unknown architecture: {arch}")
        
        module_name, class_name = self._registry[arch]
        print(f"  [LAZY LOAD] Importing {module_name}.{class_name}...")
        
        # In real vLLM: importlib.import_module(f"vllm.model_executor.models.{module_name}")
        # For demo, we'll simulate it
        time.sleep(0.1)  # Simulate import time
        
        # Cache the result
        self._cache[arch] = type(class_name, (), {})  # Simulate class
        return self._cache[arch]

# Demo
registry = LazyModelRegistry()
registry.register("LlamaForCausalLM", "llama", "LlamaForCausalLM")
registry.register("GPT2LMHeadModel", "gpt2", "GPT2LMHeadModel")
registry.register("Qwen2ForCausalLM", "qwen2", "Qwen2ForCausalLM")

print("Registry created with 3 models. Nothing imported yet!")
print("\nNow requesting LlamaForCausalLM:")
cls = registry.get_model_class("LlamaForCausalLM")
print(f"Got class: {cls}")

print("\nRequesting LlamaForCausalLM again (cached):")
cls = registry.get_model_class("LlamaForCausalLM")
print(f"Got class: {cls} (no import needed)")

print("\nGPT2 and Qwen2 were NEVER imported - lazy loading!")

## 6. Model Class Interface: Required Methods

Every model class registered with vLLM must implement a specific interface. This is the contract between the model and vLLM's execution engine.

```
+----------------------------------+
|     vLLM Model Interface         |
+----------------------------------+
| __init__(config, ...)            |
| forward(input_ids, positions,    |
|         kv_caches, attn_meta)    |
| load_weights(weights_iterator)   |
| sample(logits, sampling_meta)    | (optional - usually via LogitsSampler)
+----------------------------------+
```

In [ ]:
import torch
import torch.nn as nn
from typing import Optional, List, Iterable, Tuple, Set

class VllmModelInterface(nn.Module):
    """
    The interface that every vLLM model must implement.
    
    This is NOT an actual base class in vLLM - it's a duck-typing interface.
    Your model must have these methods with compatible signatures.
    """
    
    def __init__(
        self,
        config,           # HuggingFace model config
        cache_config=None,     # KV cache configuration
        quant_config=None,     # Quantization configuration
        lora_config=None,      # LoRA configuration (optional)
        multimodal_config=None, # Multimodal config (optional)
        prefix: str = "",      # Weight name prefix for TP
    ):
        super().__init__()
        # Initialize your model layers here
        pass
    
    def forward(
        self,
        input_ids: torch.Tensor,          # Token IDs [num_tokens]
        positions: torch.Tensor,           # Position IDs [num_tokens]
        kv_caches: List[torch.Tensor],     # KV cache for each layer
        attn_metadata,                     # Attention metadata (backend-specific)
        intermediate_tensors=None,         # For pipeline parallelism
        inputs_embeds: Optional[torch.Tensor] = None,  # For multimodal
    ) -> torch.Tensor:
        """
        Forward pass. Returns hidden states or logits.
        
        IMPORTANT: The input is NOT batched in the traditional sense.
        input_ids is [num_tokens] (1D), where num_tokens includes
        all tokens from all sequences in the batch, concatenated.
        The attn_metadata tells the attention backend how to split them.
        """
        raise NotImplementedError
    
    def load_weights(
        self,
        weights: Iterable[Tuple[str, torch.Tensor]],
    ) -> Set[str]:
        """
        Load weights from an iterator of (name, tensor) pairs.
        
        Returns the set of weight names that were loaded.
        This method handles:
        - Weight name remapping (HF -> vLLM)
        - Tensor parallelism sharding
        - Quantized weight loading
        """
        raise NotImplementedError
    
    def compute_logits(
        self,
        hidden_states: torch.Tensor,
        sampling_metadata,
    ) -> Optional[torch.Tensor]:
        """
        Compute logits from hidden states.
        Usually done via a LogitsProcessor.
        """
        raise NotImplementedError

print("Model interface methods:")
for name in ['__init__', 'forward', 'load_weights', 'compute_logits']:
    method = getattr(VllmModelInterface, name)
    print(f"  {name}: {method.__doc__[:80].strip()}...")

## 7. ModelConfig: Bridging HuggingFace and vLLM

The `ModelConfig` class in vLLM wraps the HuggingFace config and adds vLLM-specific settings. Let's understand the mapping.

In [ ]:
# Simulating ModelConfig (actual: vllm/config.py)

class SimulatedModelConfig:
    """
    Simplified version of vLLM's ModelConfig.
    
    Actual location: vllm/config.py
    
    This wraps a HuggingFace config and adds:
    - dtype selection
    - max model length
    - tensor parallel size
    - quantization settings
    """
    
    def __init__(
        self,
        model: str,                       # HuggingFace model ID or path
        dtype: str = "auto",              # Data type
        max_model_len: Optional[int] = None,  # Max sequence length
        quantization: Optional[str] = None,   # Quantization method
        trust_remote_code: bool = False,   # Allow custom code
    ):
        self.model = model
        self.dtype = dtype
        self.max_model_len = max_model_len
        self.quantization = quantization
        self.trust_remote_code = trust_remote_code
        
        # These would be loaded from HuggingFace config.json
        self.hf_config = self._load_hf_config()
    
    def _load_hf_config(self):
        """Simulate loading HuggingFace config."""
        # In reality: AutoConfig.from_pretrained(self.model)
        return {
            "architectures": ["LlamaForCausalLM"],
            "hidden_size": 4096,
            "intermediate_size": 11008,
            "num_attention_heads": 32,
            "num_hidden_layers": 32,
            "num_key_value_heads": 32,
            "vocab_size": 32000,
            "max_position_embeddings": 4096,
            "rms_norm_eps": 1e-6,
        }
    
    @property
    def architectures(self) -> List[str]:
        return self.hf_config.get("architectures", [])
    
    def get_hidden_size(self) -> int:
        return self.hf_config["hidden_size"]
    
    def get_num_layers(self) -> int:
        return self.hf_config["num_hidden_layers"]
    
    def get_head_size(self) -> int:
        return self.hf_config["hidden_size"] // self.hf_config["num_attention_heads"]

# Demo
config = SimulatedModelConfig(
    model="meta-llama/Llama-2-7b-hf",
    dtype="float16",
    max_model_len=2048,
)

print(f"Model: {config.model}")
print(f"Architecture: {config.architectures}")
print(f"Hidden size: {config.get_hidden_size()}")
print(f"Layers: {config.get_num_layers()}")
print(f"Head size: {config.get_head_size()}")

## 8. Architecture Name Mapping

A critical feature of the registry is **architecture name mapping**. Multiple HuggingFace architecture names can map to the same vLLM implementation. This is common because:

1. **Architectural similarity**: Mistral is architecturally identical to Llama
2. **Fine-tuned variants**: Different fine-tunes may declare different architecture names
3. **Version aliases**: Different versions of the same model family

In [ ]:
# Architecture name mapping examples from vLLM

ARCHITECTURE_ALIASES = {
    # Mistral reuses Llama implementation
    "MistralForCausalLM": "LlamaForCausalLM",
    
    # Aquila is also Llama-compatible
    "AquilaForCausalLM": "LlamaForCausalLM",
    
    # Yi (01.AI) uses Llama architecture
    "YiForCausalLM": "LlamaForCausalLM",
    
    # InternLM2 uses its own implementation
    "InternLM2ForCausalLM": "InternLM2ForCausalLM",
    
    # Some models have different names for the same thing
    "ChatGLMModel": "ChatGLMForCausalLM",
}

# Why does this work? Because these models share the same:
# 1. Transformer block structure
# 2. Weight naming convention
# 3. Attention mechanism

print("Architecture aliases (HF name -> vLLM implementation):")
for hf_name, vllm_name in ARCHITECTURE_ALIASES.items():
    if hf_name != vllm_name:
        print(f"  {hf_name} -> {vllm_name} (ALIASED)")
    else:
        print(f"  {hf_name} -> {vllm_name} (DIRECT)")

print("\nKey insight: If weights have the same names/shapes,")
print("and the architecture is the same, you can reuse the implementation!")

## 9. Supported Features Declaration

Models in vLLM can declare what features they support. This is done through mixins and protocol classes.

In [ ]:
from typing import Protocol, runtime_checkable

# Feature support is declared via protocol classes / mixins
# Actual implementation in vllm/model_executor/models/interfaces.py

@runtime_checkable
class SupportsLoRA(Protocol):
    """Protocol for models that support LoRA."""
    # Models must define these class attributes
    packed_modules_mapping: Dict  # Maps LoRA module names to packed weight names
    supported_lora_modules: List[str]  # List of module names that can have LoRA
    embedding_modules: Dict  # Embedding modules for LoRA
    embedding_padding_modules: List[str]  # Modules that need padding

@runtime_checkable
class SupportsPP(Protocol):
    """Protocol for models that support Pipeline Parallelism."""
    def make_empty_intermediate_tensors(
        self, batch_size: int, dtype: torch.dtype, device: torch.device
    ) -> dict:
        ...

@runtime_checkable
class SupportsMultiModal(Protocol):
    """Protocol for models that support multi-modal inputs."""
    pass

# Example: A model with LoRA and PP support
class ExampleLlamaModel(nn.Module):
    # LoRA support attributes
    packed_modules_mapping = {
        "qkv_proj": ["q_proj", "k_proj", "v_proj"],
        "gate_up_proj": ["gate_proj", "up_proj"],
    }
    supported_lora_modules = [
        "qkv_proj", "o_proj", "gate_up_proj", "down_proj",
        "embed_tokens", "lm_head",
    ]
    embedding_modules = {
        "embed_tokens": "input_embeddings",
        "lm_head": "output_embeddings",
    }
    embedding_padding_modules = ["lm_head"]
    
    def make_empty_intermediate_tensors(self, batch_size, dtype, device):
        return {"hidden_states": torch.zeros(batch_size, 4096, dtype=dtype, device=device)}

model = ExampleLlamaModel()
print(f"Supports LoRA? {isinstance(model, SupportsLoRA)}")
print(f"Supports PP? {isinstance(model, SupportsPP)}")
print(f"Supports MultiModal? {isinstance(model, SupportsMultiModal)}")
print(f"\nLoRA modules: {model.supported_lora_modules}")
print(f"Packed mappings: {model.packed_modules_mapping}")

## 10. Registration via `__init__.py`

In vLLM, models are registered in the `__init__.py` of the models package. Let's see how.

In [ ]:
# File: vllm/model_executor/models/__init__.py (simplified)

# The _GENERATION_MODELS dict maps architecture names to lazy references
_GENERATION_MODELS = {
    "LlamaForCausalLM": ("llama", "LlamaForCausalLM"),
    "MistralForCausalLM": ("llama", "LlamaForCausalLM"),
    "Qwen2ForCausalLM": ("qwen2", "Qwen2ForCausalLM"),
    # ... more models
}

# The _EMBEDDING_MODELS dict is for embedding models
_EMBEDDING_MODELS = {
    "BertModel": ("bert", "BertEmbeddingModel"),
    # ... more models
}

# Combined registry
_ALL_MODELS = {**_GENERATION_MODELS, **_EMBEDDING_MODELS}

class ModelRegistry:
    """Global model registry singleton."""
    
    _models = _ALL_MODELS
    _user_models = {}  # For custom models
    
    @classmethod
    def register_model(cls, arch_name: str, model_class):
        """Register a custom model class."""
        cls._user_models[arch_name] = model_class
    
    @classmethod
    def resolve_model_cls(cls, architectures: List[str]):
        """Find the model class for the given architecture names."""
        for arch in architectures:
            if arch in cls._user_models:
                return cls._user_models[arch]
            if arch in cls._models:
                module_name, class_name = cls._models[arch]
                # Lazy import
                module = importlib.import_module(
                    f"vllm.model_executor.models.{module_name}"
                )
                return getattr(module, class_name)
        
        raise ValueError(f"No model found for architectures: {architectures}")
    
    @classmethod
    def is_generation_model(cls, arch: str) -> bool:
        return arch in _GENERATION_MODELS or arch in cls._user_models
    
    @classmethod
    def is_embedding_model(cls, arch: str) -> bool:
        return arch in _EMBEDDING_MODELS

print("Model types:")
print(f"  Generation models: {list(_GENERATION_MODELS.keys())}")
print(f"  Embedding models: {list(_EMBEDDING_MODELS.keys())}")

## 11. The Model Loading Pipeline

Let's trace the entire flow from `LLM(model=...)` to having a running model.

```
User calls LLM(model="meta-llama/Llama-2-7b-hf")
    |
    v
1. Create ModelConfig
    - Downloads config.json from HuggingFace
    - Extracts architectures: ["LlamaForCausalLM"]
    - Determines dtype, max_len, etc.
    |
    v
2. ModelRegistry.resolve_model_cls(["LlamaForCausalLM"])
    - Looks up "LlamaForCausalLM" in registry
    - Returns LlamaForCausalLM class
    |
    v
3. Instantiate model = LlamaForCausalLM(config, ...)
    - Creates all layers (empty weights)
    - Sets up attention backends
    - Configures tensor parallelism
    |
    v
4. model.load_weights(weight_iterator)
    - Streams weights from safetensors/pytorch files
    - Handles weight name mapping
    - Shards weights for tensor parallelism
    |
    v
5. Model is ready for inference!
```

In [ ]:
# Let's simulate the full model loading pipeline

def simulate_model_loading(model_name: str):
    """Simulate the vLLM model loading pipeline."""
    
    print(f"=== Loading model: {model_name} ===")
    print()
    
    # Step 1: Create config
    print("Step 1: Creating ModelConfig")
    print(f"  - Downloading config from HuggingFace...")
    hf_config = {
        "architectures": ["LlamaForCausalLM"],
        "hidden_size": 4096,
        "num_hidden_layers": 32,
        "num_attention_heads": 32,
        "vocab_size": 32000,
    }
    arch = hf_config["architectures"][0]
    print(f"  - Architecture: {arch}")
    print()
    
    # Step 2: Resolve model class
    print("Step 2: Resolving model class")
    registry_lookup = {
        "LlamaForCausalLM": ("llama", "LlamaForCausalLM"),
    }
    module_name, class_name = registry_lookup[arch]
    print(f"  - Found: vllm.model_executor.models.{module_name}.{class_name}")
    print()
    
    # Step 3: Instantiate
    print("Step 3: Instantiating model")
    print(f"  - Creating {class_name} with {hf_config['num_hidden_layers']} layers")
    print(f"  - Hidden size: {hf_config['hidden_size']}")
    print(f"  - Attention heads: {hf_config['num_attention_heads']}")
    print(f"  - Vocab size: {hf_config['vocab_size']}")
    print()
    
    # Step 4: Load weights
    print("Step 4: Loading weights")
    param_count = (
        hf_config["hidden_size"] * hf_config["vocab_size"] +  # embeddings
        hf_config["num_hidden_layers"] * (  # per layer
            4 * hf_config["hidden_size"]**2 +  # attention
            3 * hf_config["hidden_size"] * 11008  # FFN
        )
    )
    print(f"  - Total parameters: ~{param_count / 1e9:.1f}B")
    print(f"  - Memory (FP16): ~{param_count * 2 / 1e9:.1f} GB")
    print()
    
    # Step 5: Ready
    print("Step 5: Model ready for inference!")

simulate_model_loading("meta-llama/Llama-2-7b-hf")

## 12. User Model Registration API

vLLM allows users to register custom models without modifying vLLM source code. There are two approaches:

1. **Direct registration**: `ModelRegistry.register_model(arch, cls)`
2. **Plugin system**: Using Python entry points

In [ ]:
# Approach 1: Direct Registration
# This is the simplest way to add a custom model

import torch
import torch.nn as nn

class MyCustomModel(nn.Module):
    """A toy custom model for demonstration."""
    
    def __init__(self, config, cache_config=None, quant_config=None):
        super().__init__()
        self.embed = nn.Embedding(config.vocab_size, config.hidden_size)
        self.linear = nn.Linear(config.hidden_size, config.vocab_size)
    
    def forward(self, input_ids, positions, kv_caches, attn_metadata):
        x = self.embed(input_ids)
        return self.linear(x)
    
    def load_weights(self, weights):
        params = dict(self.named_parameters())
        loaded = set()
        for name, tensor in weights:
            if name in params:
                params[name].data.copy_(tensor)
                loaded.add(name)
        return loaded

# In real vLLM, you would do:
# from vllm import ModelRegistry
# ModelRegistry.register_model("MyCustomForCausalLM", MyCustomModel)

print("Approach 1: Direct registration")
print("""  
from vllm import ModelRegistry
ModelRegistry.register_model("MyCustomForCausalLM", MyCustomModel)

# Then use it:
llm = LLM(model="path/to/model")  
# config.json must have architectures: ["MyCustomForCausalLM"]
""")

In [ ]:
# Approach 2: Plugin System (Entry Points)
# This allows packaging your model as a pip-installable plugin

SETUP_PY_EXAMPLE = """
# setup.py for a vLLM model plugin
from setuptools import setup

setup(
    name="vllm-my-custom-model",
    version="0.1.0",
    packages=["my_custom_model"],
    entry_points={
        "vllm.general_plugins": [
            "my_model = my_custom_model:register",
        ],
    },
)
"""

PLUGIN_INIT_EXAMPLE = """
# my_custom_model/__init__.py
from vllm import ModelRegistry
from .model import MyCustomForCausalLM

def register():
    ModelRegistry.register_model(
        "MyCustomForCausalLM",
        MyCustomForCausalLM
    )
"""

print("Approach 2: Plugin system")
print("\nsetup.py:")
print(SETUP_PY_EXAMPLE)
print("Plugin __init__.py:")
print(PLUGIN_INIT_EXAMPLE)

## 13. The `trust_remote_code` Pathway

Some HuggingFace models include custom model code (e.g., ChatGLM, Phi). When `trust_remote_code=True`, vLLM can load these custom classes.

In [ ]:
# How trust_remote_code works in the context of model registration

def resolve_model_with_remote_code(
    architectures: List[str],
    registry: dict,
    trust_remote_code: bool = False,
    model_path: str = ""
):
    """
    Model resolution with remote code support.
    
    Priority:
    1. Check vLLM's built-in registry
    2. If not found AND trust_remote_code=True, try HuggingFace's AutoModel
    3. If not found and trust_remote_code=False, raise error
    """
    for arch in architectures:
        # Step 1: Check built-in registry
        if arch in registry:
            print(f"  Found '{arch}' in vLLM registry")
            return registry[arch]
    
    # Step 2: Try remote code
    if trust_remote_code:
        print(f"  Architecture not in registry, loading remote code...")
        print(f"  WARNING: Running custom code from {model_path}")
        # In reality: AutoModelForCausalLM.from_pretrained(...)
        return "RemoteModelClass"
    
    # Step 3: Error
    raise ValueError(
        f"Architecture '{architectures}' not found. "
        f"Try trust_remote_code=True or register the model."
    )

# Demo: Known model
print("Case 1: Known architecture")
result = resolve_model_with_remote_code(
    ["LlamaForCausalLM"],
    {"LlamaForCausalLM": "LlamaClass"},
)
print(f"  Result: {result}")

# Demo: Unknown model with trust_remote_code
print("\nCase 2: Unknown architecture with trust_remote_code=True")
result = resolve_model_with_remote_code(
    ["CustomModelForCausalLM"],
    {"LlamaForCausalLM": "LlamaClass"},
    trust_remote_code=True,
    model_path="custom-org/custom-model"
)
print(f"  Result: {result}")

# Demo: Unknown model without trust_remote_code
print("\nCase 3: Unknown architecture without trust_remote_code")
try:
    result = resolve_model_with_remote_code(
        ["CustomModelForCausalLM"],
        {"LlamaForCausalLM": "LlamaClass"},
    )
except ValueError as e:
    print(f"  Error: {e}")

## 14. Examining the Actual vLLM Source Code

Let's look at key excerpts from the actual vLLM codebase to understand the real implementation.

In [ ]:
# Key file: vllm/model_executor/models/registry.py
# Here's what the actual ModelRegistry class looks like (annotated)

ACTUAL_REGISTRY_CODE = '''
# From vllm/model_executor/models/registry.py (simplified)

import importlib
from typing import Dict, List, Optional, Tuple, Type

# All supported models are listed here
_GENERATION_MODELS = {
    "AquilaForCausalLM": ("llama", "LlamaForCausalLM"),
    "BaiChuanForCausalLM": ("baichuan", "BaiChuanForCausalLM"),
    "BloomForCausalLM": ("bloom", "BloomForCausalLM"),
    "ChatGLMModel": ("chatglm", "ChatGLMForCausalLM"),
    "CohereForCausalLM": ("commandr", "CohereForCausalLM"),
    "DeciLMForCausalLM": ("decilm", "DeciLMForCausalLM"),
    "DeepseekForCausalLM": ("deepseek", "DeepseekForCausalLM"),
    "DeepseekV2ForCausalLM": ("deepseek_v2", "DeepseekV2ForCausalLM"),
    "FalconForCausalLM": ("falcon", "FalconForCausalLM"),
    "GemmaForCausalLM": ("gemma", "GemmaForCausalLM"),
    "Gemma2ForCausalLM": ("gemma2", "Gemma2ForCausalLM"),
    "GPT2LMHeadModel": ("gpt2", "GPT2LMHeadModel"),
    "GPTBigCodeForCausalLM": ("gpt_bigcode", "GPTBigCodeForCausalLM"),
    "GPTJForCausalLM": ("gpt_j", "GPTJForCausalLM"),
    "GPTNeoXForCausalLM": ("gpt_neox", "GPTNeoXForCausalLM"),
    "InternLMForCausalLM": ("llama", "LlamaForCausalLM"),
    "InternLM2ForCausalLM": ("internlm2", "InternLM2ForCausalLM"),
    "LlamaForCausalLM": ("llama", "LlamaForCausalLM"),
    "MistralForCausalLM": ("llama", "LlamaForCausalLM"),
    "MixtralForCausalLM": ("mixtral", "MixtralForCausalLM"),
    "OLMoForCausalLM": ("olmo", "OLMoForCausalLM"),
    "OPTForCausalLM": ("opt", "OPTForCausalLM"),
    "PhiForCausalLM": ("phi", "PhiForCausalLM"),
    "Phi3ForCausalLM": ("phi3", "Phi3ForCausalLM"),
    "Qwen2ForCausalLM": ("qwen2", "Qwen2ForCausalLM"),
    "Qwen2MoeForCausalLM": ("qwen2_moe", "Qwen2MoeForCausalLM"),
    "StableLmForCausalLM": ("stablelm", "StableLmForCausalLM"),
    # ... many more
}

_MULTIMODAL_MODELS = {
    "LlavaForConditionalGeneration": ("llava", "LlavaForConditionalGeneration"),
    "LlavaNextForConditionalGeneration": ("llava_next", "LlavaNextForConditionalGeneration"),
    "Phi3VForCausalLM": ("phi3v", "Phi3VForCausalLM"),
    # ... more multimodal models
}

_EMBEDDING_MODELS = {
    "MistralModel": ("llama_embedding", "LlamaEmbeddingModel"),
    # ... more embedding models
}

_ALL_MODELS = {
    **_GENERATION_MODELS,
    **_MULTIMODAL_MODELS,
    **_EMBEDDING_MODELS,
}
'''

print(ACTUAL_REGISTRY_CODE)

In [ ]:
# Key file: vllm/model_executor/model_loader/loader.py
# This is where model instantiation happens

MODEL_LOADER_CODE = '''
# Simplified from vllm/model_executor/model_loader/loader.py

class ModelLoader:
    """Loads and initializes models for vLLM."""
    
    def __init__(self, load_config):
        self.load_config = load_config
    
    def load_model(
        self,
        model_config,
        device_config,
        cache_config,
        parallel_config,
    ) -> nn.Module:
        """Load and return the model."""
        
        # Step 1: Get the model class
        model_class = ModelRegistry.resolve_model_cls(
            model_config.architectures
        )
        
        # Step 2: Determine quantization config
        quant_config = self._get_quant_config(
            model_config, model_class
        )
        
        # Step 3: Create the model instance
        with torch.device(device_config.device):
            model = model_class(
                config=model_config.hf_config,
                cache_config=cache_config,
                quant_config=quant_config,
            )
        
        # Step 4: Load weights
        model.load_weights(
            self._get_weights_iterator(model_config)
        )
        
        return model
'''

print(MODEL_LOADER_CODE)

## 15. How Weight Names Map Between HuggingFace and vLLM

One of the most important aspects of model registration is weight name mapping. vLLM often uses different parameter names than HuggingFace.

In [ ]:
# Weight name mapping examples

# HuggingFace Llama weight names vs vLLM weight names
HF_TO_VLLM_MAPPING = {
    # Embeddings
    "model.embed_tokens.weight": "model.embed_tokens.weight",  # Same!
    
    # Attention (per layer)
    "model.layers.{i}.self_attn.q_proj.weight": "model.layers.{i}.self_attn.qkv_proj.weight",  # Merged!
    "model.layers.{i}.self_attn.k_proj.weight": "model.layers.{i}.self_attn.qkv_proj.weight",  # Merged!
    "model.layers.{i}.self_attn.v_proj.weight": "model.layers.{i}.self_attn.qkv_proj.weight",  # Merged!
    "model.layers.{i}.self_attn.o_proj.weight": "model.layers.{i}.self_attn.o_proj.weight",  # Same
    
    # FFN (per layer)
    "model.layers.{i}.mlp.gate_proj.weight": "model.layers.{i}.mlp.gate_up_proj.weight",  # Merged!
    "model.layers.{i}.mlp.up_proj.weight": "model.layers.{i}.mlp.gate_up_proj.weight",    # Merged!
    "model.layers.{i}.mlp.down_proj.weight": "model.layers.{i}.mlp.down_proj.weight",      # Same
    
    # Layer norms
    "model.layers.{i}.input_layernorm.weight": "model.layers.{i}.input_layernorm.weight",
    "model.layers.{i}.post_attention_layernorm.weight": "model.layers.{i}.post_attention_layernorm.weight",
    
    # Output
    "model.norm.weight": "model.norm.weight",
    "lm_head.weight": "lm_head.weight",
}

print("Weight name mapping (HuggingFace -> vLLM):")
print("="*60)
for hf_name, vllm_name in HF_TO_VLLM_MAPPING.items():
    status = "SAME" if hf_name == vllm_name else "MERGED/RENAMED"
    print(f"  {status:15s}  {hf_name}")
    if hf_name != vllm_name:
        print(f"  {'':15s}  -> {vllm_name}")

print("\nKey insight: vLLM merges Q/K/V into a single QKV projection,")
print("and merges gate/up into a single gate_up projection.")
print("This allows more efficient GPU kernels.")

## 16. The `stacked_params_mapping` Pattern

vLLM uses a common pattern called `stacked_params_mapping` to handle merged weights like QKV projections.

In [ ]:
# The stacked_params_mapping pattern
# This is used in load_weights() to handle weight merging

# For Llama, the mapping tells vLLM how to merge separate HF weights
# into combined vLLM weights

stacked_params_mapping = [
    # (param_name_in_vllm, shard_name_in_hf, shard_index)
    ("qkv_proj", "q_proj", "q"),
    ("qkv_proj", "k_proj", "k"),
    ("qkv_proj", "v_proj", "v"),
    ("gate_up_proj", "gate_proj", 0),
    ("gate_up_proj", "up_proj", 1),
]

def simulate_weight_loading(hf_weights: dict, mapping: list):
    """Simulate how vLLM loads and merges weights."""
    vllm_weights = {}
    
    for hf_name, tensor in hf_weights.items():
        loaded = False
        for vllm_name, shard_name, shard_id in mapping:
            if shard_name in hf_name:
                # Replace shard name with vLLM merged name
                merged_name = hf_name.replace(shard_name, vllm_name)
                if merged_name not in vllm_weights:
                    vllm_weights[merged_name] = {}
                vllm_weights[merged_name][shard_id] = tensor
                print(f"  {hf_name} -> {merged_name}[{shard_id}]")
                loaded = True
                break
        
        if not loaded:
            vllm_weights[hf_name] = tensor
            print(f"  {hf_name} -> {hf_name} (direct copy)")
    
    return vllm_weights

# Simulate
hf_weights = {
    "model.layers.0.self_attn.q_proj.weight": torch.randn(4096, 4096),
    "model.layers.0.self_attn.k_proj.weight": torch.randn(4096, 4096),
    "model.layers.0.self_attn.v_proj.weight": torch.randn(4096, 4096),
    "model.layers.0.mlp.gate_proj.weight": torch.randn(11008, 4096),
    "model.layers.0.mlp.up_proj.weight": torch.randn(11008, 4096),
    "model.layers.0.mlp.down_proj.weight": torch.randn(4096, 11008),
    "model.layers.0.input_layernorm.weight": torch.randn(4096),
}

print("Weight merging during loading:")
print("="*60)
result = simulate_weight_loading(hf_weights, stacked_params_mapping)

## 17. Discovering What's Registered

You can inspect vLLM's registry at runtime to see all supported models.

In [ ]:
# In a vLLM environment, you can list all registered models:

DISCOVERY_CODE = '''
# Method 1: Using the ModelRegistry directly
from vllm.model_executor.models import ModelRegistry

# List all models
all_models = ModelRegistry.get_supported_archs()
print(f"Total supported architectures: {len(all_models)}")
for arch in sorted(all_models):
    print(f"  {arch}")

# Check if a specific architecture is supported
is_supported = ModelRegistry.is_supported("LlamaForCausalLM")
print(f"LlamaForCausalLM supported: {is_supported}")

# Method 2: Using the CLI
# $ python -m vllm.model_executor.models.registry
# This prints all registered models

# Method 3: Check a model's capabilities
model_cls, arch = ModelRegistry.resolve_model_cls(["LlamaForCausalLM"])
print(f"Supports LoRA: {ModelRegistry.is_lora_supported(model_cls)}")
print(f"Supports PP: {ModelRegistry.is_pp_supported(model_cls)}")
print(f"Is multimodal: {ModelRegistry.is_multimodal(model_cls)}")
'''

print(DISCOVERY_CODE)

## 18. Common Architecture Patterns

Most models in vLLM follow one of a few architectural patterns. Understanding these patterns makes it easier to add new models.

In [ ]:
# Pattern 1: Standard Decoder-Only (GPT/Llama style)
# Most common pattern. Used by Llama, Mistral, Qwen, etc.

print("Pattern 1: Standard Decoder-Only")
print("="*50)
print("""
class LlamaForCausalLM(nn.Module):
    def __init__(self, config, ...):
        self.model = LlamaModel(config, ...)  # The transformer
        self.lm_head = ParallelLMHead(...)     # Output projection
        self.logits_processor = LogitsProcessor(...)
        self.sampler = Sampler()
    
    def forward(self, input_ids, positions, kv_caches, attn_metadata):
        hidden = self.model(input_ids, positions, kv_caches, attn_metadata)
        logits = self.logits_processor(self.lm_head, hidden, ...)
        return logits
""")

print("\nPattern 2: Mixture of Experts (MoE)")
print("="*50)
print("""
class MixtralForCausalLM(nn.Module):
    # Same as decoder-only, but FFN is replaced with MoE
    # Each layer has a router + multiple expert FFNs
    
    class MixtralMoE(nn.Module):
        def __init__(self, ...):
            self.gate = nn.Linear(hidden_size, num_experts)
            self.experts = FusedMoE(...)  # Fused kernel for efficiency
""")

print("\nPattern 3: Multimodal (Vision-Language)")
print("="*50)
print("""
class LlavaForConditionalGeneration(nn.Module, SupportsMultiModal):
    def __init__(self, config, ...):
        self.vision_tower = CLIPVisionModel(...)  # Image encoder
        self.mm_projector = nn.Linear(...)         # Projection layer
        self.language_model = LlamaForCausalLM(...)  # Text decoder
    
    def forward(self, input_ids, positions, kv_caches, 
                attn_metadata, pixel_values=None):
        if pixel_values is not None:
            image_features = self.vision_tower(pixel_values)
            image_features = self.mm_projector(image_features)
            # Merge image features into input embeddings
        ...
""")

## 19. Debugging Registration Issues

Common issues when registering models and how to debug them.

In [ ]:
# Common registration issues and solutions

issues = [
    {
        "issue": "ValueError: Model architecture 'XXXForCausalLM' is not supported",
        "cause": "Architecture name in config.json doesn't match any registered name",
        "solution": [
            "1. Check the exact architecture name in config.json",
            "2. Register the model: ModelRegistry.register_model('XXX', YourClass)",
            "3. Or use trust_remote_code=True if the model has custom code",
        ]
    },
    {
        "issue": "AttributeError: 'XXXModel' has no attribute 'load_weights'",
        "cause": "Model class doesn't implement the required interface",
        "solution": [
            "1. Implement load_weights(self, weights: Iterable[Tuple[str, Tensor]])",
            "2. Make sure the method handles weight name mapping",
        ]
    },
    {
        "issue": "RuntimeError: Weight 'model.layers.0.xxx' not found",
        "cause": "Weight name mismatch between HuggingFace and vLLM model",
        "solution": [
            "1. Print both HF weight names and your model's parameter names",
            "2. Implement proper name mapping in load_weights()",
            "3. Check stacked_params_mapping for merged weights",
        ]
    },
    {
        "issue": "TypeError: forward() missing required argument 'attn_metadata'",
        "cause": "forward() signature doesn't match vLLM's expected interface",
        "solution": [
            "1. Use the exact signature:",
            "   forward(self, input_ids, positions, kv_caches, attn_metadata)",
            "2. Check for additional optional parameters",
        ]
    },
]

print("Common Registration Issues")
print("="*60)
for i, item in enumerate(issues, 1):
    print(f"\n{i}. {item['issue']}")
    print(f"   Cause: {item['cause']}")
    print(f"   Solution:")
    for s in item['solution']:
        print(f"     {s}")

## 20. Exercise: Register a Toy Model

Now it's your turn! Let's implement and register a complete (toy) model with vLLM's interface.

In [ ]:
import torch
import torch.nn as nn
from typing import Iterable, Tuple, Set, List, Optional

class ToyTransformerBlock(nn.Module):
    """A simplified transformer block."""
    
    def __init__(self, hidden_size: int, num_heads: int):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        
        # Simplified attention (merged QKV for vLLM efficiency)
        self.qkv_proj = nn.Linear(hidden_size, 3 * hidden_size, bias=False)
        self.o_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        
        # FFN (merged gate_up for vLLM efficiency)
        intermediate_size = hidden_size * 4
        self.gate_up_proj = nn.Linear(hidden_size, 2 * intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)
        
        # Norms
        self.input_layernorm = nn.LayerNorm(hidden_size)
        self.post_attention_layernorm = nn.LayerNorm(hidden_size)
    
    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        # Simplified forward (no actual attention computation)
        residual = hidden_states
        hidden_states = self.input_layernorm(hidden_states)
        
        # "Attention" (simplified)
        qkv = self.qkv_proj(hidden_states)
        hidden_states = self.o_proj(qkv[..., :self.hidden_size])
        hidden_states = residual + hidden_states
        
        # FFN
        residual = hidden_states
        hidden_states = self.post_attention_layernorm(hidden_states)
        gate_up = self.gate_up_proj(hidden_states)
        intermediate_size = gate_up.shape[-1] // 2
        gate = torch.sigmoid(gate_up[..., :intermediate_size])
        up = gate_up[..., intermediate_size:]
        hidden_states = self.down_proj(gate * up)
        hidden_states = residual + hidden_states
        
        return hidden_states


class ToyForCausalLM(nn.Module):
    """
    A toy model implementing the vLLM model interface.
    
    This model follows the standard decoder-only pattern
    and can be registered with vLLM's ModelRegistry.
    """
    
    def __init__(self, config, cache_config=None, quant_config=None):
        super().__init__()
        self.config = config
        
        # Embedding
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        
        # Transformer blocks
        self.layers = nn.ModuleList([
            ToyTransformerBlock(config.hidden_size, config.num_attention_heads)
            for _ in range(config.num_hidden_layers)
        ])
        
        # Final norm
        self.norm = nn.LayerNorm(config.hidden_size)
        
        # Output head
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
    
    def forward(
        self,
        input_ids: torch.Tensor,
        positions: torch.Tensor,
        kv_caches: List[torch.Tensor],
        attn_metadata=None,
    ) -> torch.Tensor:
        """Forward pass implementing vLLM's interface."""
        # Embed tokens
        hidden_states = self.embed_tokens(input_ids)
        
        # Process through transformer blocks
        for layer in self.layers:
            hidden_states = layer(hidden_states)
        
        # Final norm and output
        hidden_states = self.norm(hidden_states)
        logits = self.lm_head(hidden_states)
        
        return logits
    
    def load_weights(self, weights: Iterable[Tuple[str, torch.Tensor]]) -> Set[str]:
        """Load weights from HuggingFace format."""
        # Define weight name mapping
        stacked_params_mapping = [
            ("qkv_proj", "q_proj", "q"),
            ("qkv_proj", "k_proj", "k"),
            ("qkv_proj", "v_proj", "v"),
            ("gate_up_proj", "gate_proj", 0),
            ("gate_up_proj", "up_proj", 1),
        ]
        
        params = dict(self.named_parameters())
        loaded = set()
        
        for name, tensor in weights:
            # Check if this weight needs merging
            is_stacked = False
            for vllm_name, hf_name, shard_id in stacked_params_mapping:
                if hf_name in name:
                    merged_name = name.replace(hf_name, vllm_name)
                    if merged_name in params:
                        # Copy shard into merged weight
                        print(f"  Merging {name} -> {merged_name}[{shard_id}]")
                        loaded.add(name)
                        is_stacked = True
                        break
            
            if not is_stacked and name in params:
                params[name].data.copy_(tensor)
                loaded.add(name)
                print(f"  Loaded {name}")
        
        return loaded


# Create a config object
class ToyConfig:
    vocab_size = 1000
    hidden_size = 256
    num_attention_heads = 4
    num_hidden_layers = 2

# Instantiate the model
config = ToyConfig()
model = ToyForCausalLM(config)

print("ToyForCausalLM created successfully!")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"\nModel structure:")
for name, param in model.named_parameters():
    print(f"  {name}: {list(param.shape)}")

In [ ]:
# Test the toy model with a forward pass

# Simulate input
batch_size = 4
seq_len = 16
input_ids = torch.randint(0, config.vocab_size, (batch_size, seq_len))
positions = torch.arange(seq_len).unsqueeze(0).expand(batch_size, -1)

# Forward pass
with torch.no_grad():
    logits = model(input_ids, positions, kv_caches=[], attn_metadata=None)

print(f"Input shape: {input_ids.shape}")
print(f"Output shape: {logits.shape}")
print(f"Expected: [{batch_size}, {seq_len}, {config.vocab_size}]")
print(f"\nModel works correctly!")

In [ ]:
# Simulate registering the model with vLLM

# Create a mock registry
mock_registry = {}

def register_model(arch_name: str, model_class: type):
    """Register a model class."""
    # Validate interface
    required_methods = ['forward', 'load_weights']
    for method in required_methods:
        if not hasattr(model_class, method):
            raise ValueError(f"Model class must implement {method}()")
    
    mock_registry[arch_name] = model_class
    print(f"Successfully registered '{arch_name}' -> {model_class.__name__}")
    print(f"  Has forward(): {hasattr(model_class, 'forward')}")
    print(f"  Has load_weights(): {hasattr(model_class, 'load_weights')}")

# Register our toy model
register_model("ToyForCausalLM", ToyForCausalLM)

# Verify it's registered
print(f"\nRegistered models: {list(mock_registry.keys())}")

# Resolve and instantiate
ModelClass = mock_registry["ToyForCausalLM"]
model = ModelClass(ToyConfig())
print(f"\nInstantiated: {type(model).__name__}")

## 21. Exercises

### Exercise 1: Add Architecture Aliasing
Modify the `ToyForCausalLM` to also be accessible under the architecture name `"ToyV2ForCausalLM"` (imagine ToyV2 has the same architecture but a different name in `config.json`).

### Exercise 2: Feature Declaration
Add `SupportsLoRA` and `SupportsPP` protocol support to `ToyForCausalLM` by implementing the required attributes and methods.

### Exercise 3: Weight Name Mapping
Given the following HuggingFace weight names, write the `stacked_params_mapping` and `load_weights()` logic:
```
model.layers.0.attention.query.weight
model.layers.0.attention.key.weight  
model.layers.0.attention.value.weight
model.layers.0.ffn.w1.weight
model.layers.0.ffn.w2.weight
model.layers.0.ffn.w3.weight
```
Map them to vLLM merged weights:
```
model.layers.0.attention.qkv_proj.weight  (merged q+k+v)
model.layers.0.ffn.gate_up_proj.weight    (merged w1+w3)
model.layers.0.ffn.down_proj.weight       (w2 directly)
```

### Exercise 4: Plugin Package
Create the file structure for a pip-installable vLLM model plugin:
```
vllm-toy-model/
  setup.py
  toy_model/
    __init__.py
    model.py
    config.py
```

In [ ]:
# Exercise 1 Solution Skeleton

# Register under multiple names
def register_with_aliases(arch_names: List[str], model_class: type, registry: dict):
    """Register a model under multiple architecture names."""
    for name in arch_names:
        registry[name] = model_class
        print(f"Registered: {name} -> {model_class.__name__}")

# TODO: Call this with ["ToyForCausalLM", "ToyV2ForCausalLM"]
# register_with_aliases(["ToyForCausalLM", "ToyV2ForCausalLM"], ToyForCausalLM, mock_registry)

print("Exercise 1: Implement architecture aliasing")
print("Hint: Register the same class under multiple architecture names")

## Summary

In this notebook, we covered:

1. **ModelRegistry**: The central mapping from architecture names to Python classes
2. **Lazy loading**: Models are only imported when needed for fast startup
3. **Model interface**: `__init__`, `forward`, `load_weights` are required methods
4. **ModelConfig**: Wraps HuggingFace config with vLLM-specific settings
5. **Architecture aliasing**: Multiple HF names can map to one vLLM class
6. **Feature protocols**: `SupportsLoRA`, `SupportsPP`, `SupportsMultiModal`
7. **Weight mapping**: HF weights are merged (QKV, gate_up) for efficiency
8. **Registration methods**: Direct registration and plugin system

### Next Steps
In Chapter 8.2, we'll implement a complete new model in vLLM from scratch, with all the attention backend integration and tensor parallelism support.